# TheTelos — Sesli Özet Üretici

1. Üst menü: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**
2. Üst menü: **Çalışma zamanı → Tümünü çalıştır**

Bu kadar. Kapanırsa 2. adımı tekrarla, kaldığı yerden devam eder.
Önce AYARLAR hücresindeki boş tırnakları (TLS_KEY, R2_KEY, R2_SECRET) doldurun.

In [ ]:
# KURULUM — Colab Python 3.13 kullanıyor, Kokoro 3.12 istiyor → ayrı bir 3.12 ortamı kurulur (2-4 dk)
!apt-get -qq -y install espeak-ng ffmpeg > /dev/null
!pip install -q uv
!uv venv /content/k -p 3.12 -q
!uv pip install -q --python /content/k/bin/python pip "kokoro>=0.9.4" soundfile boto3 requests numpy
!/content/k/bin/python -c "import kokoro; print('Kokoro kuruldu ✓')"

In [ ]:
# AYARLAR
TLS_KEY = ""   # ← panel > Araçlar > Sesli Özet sayfasındaki kod

SITE        = "https://thetelos.org/thetelos-panel/api/tts.php"
PUBLIC_BASE = "https://audio.thetelos.org"
VOICE       = "am_michael"     # erkek, sakin anlatıcı. Kadın sesi için: "af_heart"
SPEED       = 1.0

R2_ACCOUNT = "3da5d617f59e35f9aeddbbef12785676"
R2_KEY     = ""   # ← R2 Access Key ID
R2_SECRET  = ""   # ← R2 Secret Access Key
R2_BUCKET  = "thetelos-audio"

import json, requests
json.dump(dict(TLS_KEY=TLS_KEY, SITE=SITE, PUBLIC_BASE=PUBLIC_BASE, VOICE=VOICE, SPEED=SPEED, R2_ACCOUNT=R2_ACCOUNT, R2_KEY=R2_KEY, R2_SECRET=R2_SECRET, R2_BUCKET=R2_BUCKET), open("/content/cfg.json", "w"))
_r = requests.get(SITE, params={"action": "stats", "key": TLS_KEY}, timeout=60)
print("Site bağlantısı tamam ✓" if _r.status_code == 200 else f"Site anahtarı hatalı ✗ ({_r.status_code}) — bana ekran görüntüsü at")

In [ ]:
%%writefile /content/run.py
import json, requests, boto3, subprocess, numpy as np, soundfile as sf, time
from kokoro import KPipeline
C = json.load(open("/content/cfg.json"))
pipe = KPipeline(lang_code="a")
s3 = boto3.client("s3", endpoint_url=f"https://{C['R2_ACCOUNT']}.r2.cloudflarestorage.com",
                  aws_access_key_id=C["R2_KEY"], aws_secret_access_key=C["R2_SECRET"], region_name="auto")

def speak(text):
    parts = []
    for _, _, audio in pipe(text, voice=C["VOICE"], speed=C["SPEED"], split_pattern=r"\n+"):
        if audio is not None:
            parts.append(audio.numpy() if hasattr(audio, "numpy") else audio)
            parts.append(np.zeros(int(24000 * 0.35), dtype=np.float32))
    return np.concatenate(parts) if parts else None

done = 0
while True:
    r = requests.get(C["SITE"], params={"action": "list", "key": C["TLS_KEY"], "limit": 5}, timeout=120).json()
    items = r.get("items", [])
    if not items:
        print("Seslendirilecek kitap kalmadı ✓", flush=True); break
    for it in items:
        t0 = time.time(); pid = it["post_id"]
        audio = speak(it["text"])
        if audio is None:
            print("boş ses, atlandı:", pid, flush=True); continue
        sf.write("/tmp/a.wav", audio, 24000)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/tmp/a.wav", "-ac", "1", "-b:a", "64k", "/tmp/a.mp3"], check=True)
        key = f"audio/{pid}.mp3"
        s3.upload_file("/tmp/a.mp3", C["R2_BUCKET"], key, ExtraArgs={"ContentType": "audio/mpeg", "CacheControl": "public, max-age=31536000"})
        secs = int(len(audio) / 24000)
        a = requests.post(C["SITE"], data={"action": "attach", "key": C["TLS_KEY"], "post_id": pid, "url": f"{C['PUBLIC_BASE']}/{key}", "seconds": secs}, timeout=60).json()
        done += 1
        print(f"✓ {done}. {it['title'][:60]} — {secs//60} dk ses, {int(time.time()-t0)} sn", "" if a.get("ok") else a, flush=True)

In [ ]:
# ÇALIŞTIR — kitaplar tek tek seslendirilir ve siteye eklenir
!/content/k/bin/python -u /content/run.py